In [ ]:
import pycountry
import json
from os import listdir as ls
from datetime import datetime
from matplotlib import pyplot as plt 

from emu_renewal.inputs import DATA_PATH, get_apple_mobility, get_worldbank_national_pop, \
    get_country_vacc_data, get_indicator_series_from_who_data, get_country_vacc_data, get_undesa_national_pop, \
    find_null_data, find_neg_data, has_change_repeated, has_outlier
from emu_renewal.run import find_run_end_time

In [ ]:
def get_mob_countries():
    gmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
    fbmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
    amob_avail = []
    amob_unavail = []
    for c in pycountry.countries:
        try:
            iso3 = c.alpha_3
            get_apple_mobility(iso3)
            amob_avail.append(iso3)
        except:
            amob_unavail.append(iso3)
    return list(set(amob_avail + fbmob_avail + gmob_avail))

# json.dump(get_mob_countries(), open(DATA_PATH / "config/mob_countries.json", "w"))
any_mob_avail = json.load(open(DATA_PATH / "config/mob_countries.json", "r"))

In [ ]:
# Gather data
start_time = datetime(2020, 4, 1)
case_data = {}
deaths_data = {}
for c in any_mob_avail:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, 0.05, c)
    data = get_indicator_series_from_who_data("New_cases", c)
    date_filter = (start_time < data.index) & (data.index < end_time)
    case_data[c] = data[date_filter]
    data = get_indicator_series_from_who_data("New_deaths", c)[date_filter]
    date_filter = (start_time < data.index) & (data.index < end_time)
    deaths_data[c] = data[date_filter]

In [ ]:
def is_mostly_zeros(data):
    if len(data) > 0:
        return sum((data == 0.0).astype(int)) / len(data) > 0.5
    else:
        return False

In [ ]:
n_repeats = 6
no_cases = find_null_data(case_data)
no_deaths = find_null_data(deaths_data)
neg_cases = find_neg_data(case_data)
neg_deaths = find_neg_data(deaths_data)

cases_repeat = [c for c, d in case_data.items() if has_change_repeated(d, n_repeats)]
deaths_repeat = [c for c, d in deaths_data.items() if has_change_repeated(d, n_repeats)]
case_outliers = [c for c, d in case_data.items() if has_outlier(d)]
death_outliers = [c for c, d in deaths_data.items() if has_outlier(d)]
mostly_zeroes = [c for c, d in case_data.items() if is_mostly_zeros(d)]

In [ ]:
excluded = list(set(no_cases + no_deaths + neg_cases + neg_deaths + cases_repeat + deaths_repeat + case_outliers + death_outliers + mostly_zeroes))
included = [c for c in any_mob_avail if c not in excluded]
included.append("AUS")

In [ ]:
json.dump(included, open(DATA_PATH / "config/included.json", "w"))
json.dump(excluded, open(DATA_PATH / "config/excluded.json", "w"))

In [ ]:
def plot_cases_deaths(country, c_data, d_data, pop, title):
    def pop_normalise(data):
        return data / pop
    def pop_denormalise(data):
        return data * pop
    country_name = pycountry.countries.lookup(country).name
    fig, axes = plt.subplots(2, 1, sharex=True)
    top_ax = axes[0]
    top_ax.plot(c_data.index, c_data, marker="o", linewidth=0.0, markersize=3.0)
    top_ax.set_title(f"cases, {country_name} ({country})")
    secondary_ax = top_ax.secondary_yaxis("right", functions=(pop_normalise, pop_denormalise))
    top_ax.set_ylabel("number")
    secondary_ax.set_ylabel("per capita")
    bottom_ax = axes[1]
    bottom_ax.plot(d_data.index, d_data, marker="o", linewidth=0.0, markersize=3.0)
    bottom_ax.set_title(f"deaths, {country_name} ({country})")
    secondary_ax = bottom_ax.secondary_yaxis("right", functions=(pop_normalise, pop_denormalise))
    bottom_ax.set_ylabel("number")
    secondary_ax.set_ylabel("per capita")
    plt.setp(bottom_ax.xaxis.get_majorticklabels(), rotation=70)
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    fig.savefig(f"indicators_{country}.png")

In [ ]:
pops = {}
for c in any_mob_avail:
    try:
        pops[c] = get_worldbank_national_pop(c, 2020)
    except:
        pops[c] = get_undesa_national_pop(c)

for c in included:
    plot_cases_deaths(c, case_data[c], deaths_data[c], pops[c], "--included--")
for c in excluded:
    plot_cases_deaths(c, case_data[c], deaths_data[c], pops[c], "--excluded--")